# 1. Initializations

## 1.1 General CPU/GPU Checks (NVIDIA cards)

In [ ]:
### global
import logging
import os
import time
from datetime import datetime
from smartcheck.logger_config import setup_logger

setup_logger(logging.INFO)
print(f'Path [{os.environ["PATH"]}]')

# Test pytorch GPU config
import torch
cuda_test = torch.cuda.is_available()
print(f"✅ Torch CUDA available: {cuda_test}")
device_name_gpu = torch.cuda.get_device_name(0)
device_gpu = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Device Name: {device_name_gpu} | Device reference: {device_gpu}")

# 🧠 Benchmark GPU
if torch.cuda.is_available():
    print("Lancement benchmark sur GPU...")
    start = time.time()
    a = torch.randn(10000, 10000, device=device_gpu)
    b = torch.randn(10000, 10000, device=device_gpu)
    c = torch.matmul(a, b)
    torch.cuda.synchronize()  # 🔁 Synchronisation obligatoire pour mesure fiable
    print("Fin multiplication")
    print("Durée:", time.time() - start, "secondes")

# 🧠 Benchmark CPU
device_cpu = torch.device("cpu")
print("Lancement benchmark sur CPU...")
start = time.time()
a = torch.randn(10000, 10000, device=device_cpu)
b = torch.randn(10000, 10000, device=device_cpu)
c = torch.matmul(a, b)
print("Fin multiplication")
print("Durée:", time.time() - start, "secondes")

## 1.2 General imports

In [ ]:
# Pour la manipulation de tableaux et Dataframes
import numpy as np
import pandas as pd

# modelisation
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import torch
from torch.utils.data import DataLoader, Dataset
from torchsummary import summary
from torch.utils.tensorboard.writer import SummaryWriter
from tqdm.notebook import tqdm

# Pour la visualisation des performances
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

In [ ]:
import smartcheck.dataframe_common as dfc

# 2. Loading and Data Enrichment

In [ ]:
df_wq_raw = dfc.load_dataset_from_config('wine_quality_data', sep=',')

if df_wq_raw is not None and isinstance(df_wq_raw, pd.DataFrame):
    df_wq = df_wq_raw.copy()

In [ ]:
df_wq.info()

In [ ]:
X = df_wq.drop(['quality'], axis=1).values
y = df_wq['quality'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=1234)

# 3. Deep learning

## 3.1 Modèle basé sur Layer PyTorch simples

#### Creation & Execution unitaire (descente de gradient en regression linéaire simple)

In [ ]:
class CustomLinearLayer(torch.nn.Module):
    def __init__(self, input_dim, output_dim):
        super(CustomLinearLayer, self).__init__()
        # Initialisation des paramètres (poids et biais)
        self.W = torch.nn.Parameter(torch.randn(input_dim, output_dim))
        self.b = torch.nn.Parameter(torch.randn(output_dim))

    def forward(self, x):
        # Définir la sortie
        return x @ self.W + self.b
    
customLinearRegression = CustomLinearLayer(X.shape[1], 1)

In [ ]:
# Visualisation des paramètresq du layer
print(list(customLinearRegression.parameters()))

In [ ]:
# Remise à zéro des coefficients W et b du gradient
customLinearRegression.zero_grad()
print(list(customLinearRegression.parameters()))

In [ ]:
# Verification de l'état du layer (actif/inactif)
print(f'La couche {customLinearRegression._get_name()} est {"active" if customLinearRegression.training else "inactive"}')

In [ ]:
# Mise en forme tensor de X_train et y_train et calcul de la prediction sur x_train 
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
# X_train_tensor = torch.tensor(X_train).to(dtype=torch.float32)
# X_train_tensor = torch.tensor(X_train).float()
display(X_train_tensor)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
display(y_train_tensor)
y_train_hat = customLinearRegression(X_train_tensor)
display(y_train_hat)

In [ ]:
# Definition de la fonction de perte et calcul sur le tensor y_train_hat
loss_func = torch.nn.MSELoss()
assert y_train_hat.shape == y_train_tensor.shape, (
    f"Incompatible shapes: {y_train_hat.shape} vs {y_train_tensor.shape}"
)
loss_value = loss_func(y_train_hat, y_train_tensor)

In [ ]:
### Cycle unitaire sur y_train_hat par l'intermédiare du gradient calculé sur loss_value
# Reset des paramètres du gradient (W et b ici)
customLinearRegression.zero_grad()
# Calcul du gradient
loss_value.backward()
# Instanciation de l'optimizer adam
optimizer = torch.optim.Adam(params=customLinearRegression.parameters(), lr = 1e-1)
# Descente de gradient
print(f"Before Optimizer : {customLinearRegression.W}")
optimizer.step()
print(f"After Optimizer : {customLinearRegression.W}")

#### Entraînement et métriques

In [ ]:
# Nombre d'époques (itération de descente de gradient)
nb_epoch = 100
# Fonction de perte
loss_func = torch.nn.MSELoss()
# Mise au format tensor de la variable cible
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)

for i in range(nb_epoch):
    # Gradient à zéros
    customLinearRegression.zero_grad()
    # Sortie du modèle
    y_train_hat = customLinearRegression(X_train_tensor)
    # Calcul de la fonction de perte
    loss_value = loss_func(y_train_hat, y_train_tensor)
    # Calcul du gradient
    loss_value.backward()
    # Descente de gradient.
    optimizer.step()
    print(f"Loss for the epoch {i} : {loss_value.item()}")

#### Prédiction et évaluation

In [ ]:
# Evaluation de l'accuracy
def evaluate(X, y):
    # Passer le modèle en évaluation
    customLinearRegression.eval()
    with torch.no_grad():
        # Prédiction du modèle pour un batch donné
        y_pred = customLinearRegression(X)
        # Calcul de la fonction de perte pour l'utiliser comme une métrique
        loss = loss_func(y_pred, y)
    # Ensemble des prédictions du jeu de données
    predictions = y_pred.cpu().numpy()
    # Ensemble des vraies valeurs du jeu de données
    true_vals = y.cpu().numpy()
    # Calcul de la MAE
    mae = mean_absolute_error(true_vals, predictions)

    return loss.item(), mean_absolute_error(true_vals, predictions)

loss_val, mae_val = evaluate(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32).view(-1, 1)
)
print(f"Test loss (MSE): {loss_val:.4f}, MAE: {mae_val:.4f}")

## 3.2 Modèle basé sur Layer PyTorch avec batch

#### Creation des batch de données

In [ ]:
# Application a une problématique temporelle
class TimeSeriesWindowDataset(Dataset):
    def __init__(self, data, window_size):
        self.data = data
        # la fenetre temporelle devrait porter la saisonnalité (ie. es window_size précédentes valeur permettent de prédire la valeur actuelle)
        self.window_size = window_size

    def __getitem__(self, idx):
        # Pour chaque donnée, retourner un tuple (x, y) basé sur la fenetre temporelle
        x = self.data[idx:idx+self.window_size]
        y = self.data[idx+self.window_size]
        return (x, y)

    def __len__(self):
        # la taille est calée sur la fenetre temporelle
        return len(self.data) - self.window_size

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, X, y):
        assert len(X) == len(y), "X and y must have same length"
        self.X = X
        self.y = y

    def __getitem__(self, idx):
        # Pour chaque donnée, retourner un tuple (x, y)
        x = self.X[idx]
        y = self.y[idx]
        return (x, y)

    def __len__(self):
        # La taille est calée sur la dimension de X
        return len(self.X)

# Chargement du train set batch
train_set = CustomDataset(X_train, y_train)
train_loader = DataLoader(train_set, batch_size=200, shuffle=True)

# Chargement du test set batch
test_set = CustomDataset(X_test, y_test)
test_loader = DataLoader(test_set, batch_size=100, shuffle=False)

# Affichage du premier batch train
print(next(iter(train_loader)))

#### Creation du modèle en couche séquentielle

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device utilisé:", device)

nn_pytorch_seq = torch.nn.Sequential(
   torch.nn.Linear(X.shape[1], 100),
   torch.nn.ReLU(),
   torch.nn.Linear(100, 100),
   torch.nn.ReLU(),
   torch.nn.Linear(100, 100),
   torch.nn.ReLU(),
   torch.nn.Linear(100, 50),
   torch.nn.ReLU(),
   torch.nn.Linear(50, 1)
)
nn_pytorch_seq.to(device)

In [ ]:
summary(nn_pytorch_seq, input_size=(X.shape[-1],), device=str(device))

#### Entraînement et métriques

In [ ]:
# Initialisation du writer TensorBoard avec horodatage pour différencier les runs
log_dir = f"../../runs/nn_pytorch_seq_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
writer = SummaryWriter(log_dir=log_dir)
example_batch = next(iter(train_loader))[0].to(device).float()
writer.add_graph(nn_pytorch_seq, example_batch)
# Nombre d'époques (itération de descente de gradient)
nb_epoch = 100
# Définition de l'optimizer
optimizer = torch.optim.Adam(nn_pytorch_seq.parameters(), 1e-3)
# Définition de la fonction de perte
loss_func = torch.nn.MSELoss()
for epoch in range(nb_epoch):
    # Passer le modèle en mode train
    nn_pytorch_seq.train()
    # On cherche a déterminer la perte totale comme cumul de celle d'application de la prédiction à tous les batches de données
    loss_total = 0
    progress_bar = tqdm(
        enumerate(train_loader),  # utiliser enumerate directement ici
        total=len(train_loader),  # important pour fixer le total
        desc=f"Epoch {epoch:1d}", 
        leave=True, 
        disable=False
    )
    for i, batch in progress_bar:
        # récupération des tensor explicatif et cible du batch de données
        X_batch, y_batch = batch
        # Affectation au device cible (cpu ou cuda avec gpu)
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        # Remise à zéro du gradient
        nn_pytorch_seq.zero_grad()
        # Calcul de prédiction
        y_pred = nn_pytorch_seq(X_batch.to(torch.float32))[:,0]
        # Calcul de la fonction de perte
        loss_batch = loss_func(y_pred*100, y_batch.to(torch.float32)*100) #torch.mean(torch.abs(y_pred- y_batch.to(torch.float32)))#
        # Calculer le gradient de la loss en fonction de chaque couche
        loss_batch.backward()
        # Clipper le gradient entre 0 et 1 pour plus de stabilité
        torch.nn.utils.clip_grad_norm_(nn_pytorch_seq.parameters(), max_norm=1.0)
        # Descente de gradient et actualisation des paramètres
        optimizer.step()
        # Participation de la perte du batch à la perte totale
        loss_total += loss_batch.item() 
        avg_loss = loss_total / (i + 1)        
        progress_bar.set_postfix(training_loss=f"{avg_loss:.2e}")  # format scientifique lisible
    loss_epoch = loss_total / len(train_loader)  
    print(f"Epoch : {epoch+1}/{nb_epoch} -- Training loss {loss_epoch}")
    # Ajout de la loss à TensorBoard
    writer.add_scalar("Loss/train", loss_epoch, epoch+1)
# Libération du writer pour alimenter tensorboard
writer.close()

#### Prédiction et évaluation

In [ ]:
def evaluate(model, dataloader_val, criterion, device, target_scaler=None):
    # Passer en mode évaluation du modèle
    model.eval()
    # Initialiser la perte totale / predictions / valeurs réelles
    loss_val_total = 0
    predictions, true_vals = [], []
    # Parcourir tous les batches pour récupérer les prédictions/valeurs réelles
    for X_batch, y_batch in dataloader_val:
        X_batch = X_batch.to(device).float()
        y_batch = y_batch.to(device).float().view(-1, 1)
        # Pas de calcul de gradient nécessaire pendant la prédiction
        with torch.no_grad():
            y_pred = model(X_batch)
        # Calcul de la perte pour ce batch
        loss = criterion(y_pred, y_batch)
        loss_val_total += loss.item()
        # Conversion en numpy et stockage pour le calcul des métriques (scikit ne tolère pas les tensors)
        y_pred_np = y_pred.detach().cpu().numpy()
        y_true_np = y_batch.cpu().numpy()
        predictions.append(y_pred_np)
        true_vals.append(y_true_np)
    # Concaténer tous les batches
    predictions = np.concatenate(predictions).ravel()
    true_vals = np.concatenate(true_vals).ravel()
    # Inverser un éventuel scaler (StandardScaler, MinMaxScaler, etc.)
    if target_scaler is not None:
        predictions = target_scaler.inverse_transform(predictions.reshape(-1, 1)).ravel()
        true_vals = target_scaler.inverse_transform(true_vals.reshape(-1, 1)).ravel()
    # Calcul de la moyenne de la loss
    loss_val_avg = loss_val_total / len(dataloader_val)
    # Calcul des métriques
    metrics = {
        "mae": mean_absolute_error(true_vals, predictions),
        "mse": mean_squared_error(true_vals, predictions),
        "rmse": np.sqrt(mean_squared_error(true_vals, predictions)),
    }
    return loss_val_avg, metrics, predictions, true_vals

loss, metrics, y_pred, y_true = evaluate(
    model=nn_pytorch_seq,
    dataloader_val=test_loader,
    criterion=torch.nn.MSELoss(),
    device=device,
    target_scaler=None  # ou scaler_y si y avait été transformé
)

print(f"Loss: {loss:.4f}")
for k, v in metrics.items():
    print(f"{k.upper()}: {v:.4f}")

In [ ]:
# S'assurer qu'on a bien des tableaux 1D
y_pred = np.array(y_pred).ravel()
y_true = np.array(y_true).ravel()

# Créer un DataFrame pour une compatibilité propre avec seaborn
df_plot = pd.DataFrame({
    'True': y_true,
    'Pred': y_pred
})

# Option : arrondir ou regrouper les valeurs cibles si elles sont trop nombreuses
# df_plot['True'] = df_plot['True'].round(1)

# Violinplot
fig, ax = plt.subplots(figsize=(15, 8))
sns.violinplot(data=df_plot, x='True', y='Pred', ax=ax)

ax.set_xlabel("Valeurs réelles")
ax.set_ylabel("Prédictions")
plt.title("Distribution des prédictions selon les vraies valeurs")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()